In [1]:
import os, json, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch, torch.nn.functional as F
import torch.nn as nn
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, auc, precision_recall_fscore_support
)
import matplotlib.pyplot as plt
import seaborn as sns

# --------- EDIT only if your paths differ ----------
PROJECT_ROOT = Path("/Users/syedadnanahmad/Downloads/AI_TUMOUR_detection")
EMB_ROOT     = PROJECT_ROOT / "embeddings"
IDX_CSV      = PROJECT_ROOT / "notebooks" / "slide_index.csv"
MODEL_DIR    = PROJECT_ROOT / "models"
RESULTS_DIR  = PROJECT_ROOT / "results"
K = 20   # use same K as final training
# ---------------------------------------------------

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# device
device = torch.device("mps") if torch.backends.mps.is_available() else (torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print("Device:", device)

# load index
df_index = pd.read_csv(IDX_CSV)

OSCC_CLASSES = ["wdoscc","mdoscc","pdoscc"]
ALL_CLASSES_5 = ["normal","osmf","wdoscc","mdoscc","pdoscc"]   # if you have 5-class model


Device: mps


In [2]:
# ----------------- Models definitions (must match training) -----------------
class TopK_AttentionMIL(nn.Module):
    def __init__(self, emb_dim=512, hidden_dim=512, num_classes=3, k=20):
        super().__init__()
        self.k = k
        self.V = nn.Linear(emb_dim, hidden_dim)
        self.U = nn.Linear(emb_dim, hidden_dim)
        self.w = nn.Linear(hidden_dim, 1)
        self.classifier = nn.Sequential(
            nn.Linear(emb_dim, emb_dim//2),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(emb_dim//2, num_classes)
        )
    def forward(self, H):
        Vh = torch.tanh(self.V(H))
        Uh = torch.sigmoid(self.U(H))
        A = self.w(Vh * Uh).squeeze(1)
        Ksel = min(self.k, H.shape[0])
        top_vals, top_idx = torch.topk(A, Ksel)
        H_top = H[top_idx]
        A_top = torch.softmax(top_vals, dim=0)
        agg = torch.sum(A_top.unsqueeze(1) * H_top, dim=0)
        logits = self.classifier(agg)
        return logits.unsqueeze(0), A_top.detach().cpu().numpy(), top_idx.detach().cpu().numpy()

In [3]:
# ----------------- Helpers -----------------
def load_npy(path):
    return np.load(path)

def sample_K_from_emb(np_arr, k=K):
    n = np_arr.shape[0]
    if n >= k:
        sel = np.random.choice(n, k, replace=False)
    else:
        sel = np.random.choice(n, k, replace=True)
    return np_arr[sel]

def safe_roc_auc(y_true, y_score, n_classes, labels):
    """Compute one-vs-rest ROC AUCs; handle classes absent in y_true."""
    aucs = {}
    try:
        y_true_bin = np.zeros((len(y_true), n_classes))
        for i, l in enumerate(labels):
            y_true_bin[:, i] = (np.array(y_true) == l).astype(int)
        # y_score expected shape (N, n_classes)
        for i, lab in enumerate(labels):
            if y_true_bin[:, i].sum() == 0:
                aucs[lab] = None
            else:
                aucs[lab] = roc_auc_score(y_true_bin[:, i], y_score[:, i])
    except Exception as e:
        print("ROC compute failed:", e)
        for lab in labels:
            aucs[lab] = None
    return aucs

def plot_and_save_cm(cm, classes, outpath):
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.savefig(outpath, bbox_inches="tight"); plt.close()

def plot_roc_curves(y_true, y_score, labels, outpath):
    n = len(labels)
    # one-vs-rest binarize
    y_true_bin = np.zeros((len(y_true), n))
    for i, l in enumerate(labels):
        y_true_bin[:, i] = (np.array(y_true) == l).astype(int)
    plt.figure(figsize=(6,5))
    for i, lab in enumerate(labels):
        if y_true_bin[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{lab} (AUC={roc_auc:.2f})")
    plt.plot([0,1],[0,1],"k--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.legend(loc="lower right")
    plt.title("One-vs-Rest ROC curves")
    plt.savefig(outpath, bbox_inches="tight"); plt.close()

In [4]:
stage1_path = MODEL_DIR / "stage1_balanced_best.pth"
stage2_path = MODEL_DIR / f"oscc_topkK{K}_best.pth"
model_stage1 = None
model_stage2 = None

if stage1_path.exists():
    try:
        m = TopK_AttentionMIL(emb_dim=512, hidden_dim=256, num_classes=2, k=K).to(device)
        ck = torch.load(stage1_path, map_location=device)
        if isinstance(ck, dict) and "model_state" in ck:
            m.load_state_dict(ck["model_state"])
        else:
            m.load_state_dict(ck)
        m.eval(); model_stage1 = m
        print("Loaded Stage-1:", stage1_path)
    except Exception as e:
        print("Could not load stage1:", e)

if stage2_path.exists():
    try:
        m2 = TopK_AttentionMIL(emb_dim=512, hidden_dim=512, num_classes=3, k=K).to(device)
        ck2 = torch.load(stage2_path, map_location=device)
        if isinstance(ck2, dict) and "model_state" in ck2:
            m2.load_state_dict(ck2["model_state"])
        else:
            m2.load_state_dict(ck2)
        m2.eval(); model_stage2 = m2
        print("Loaded Stage-2:", stage2_path)
    except Exception as e:
        print("Could not load stage2:", e)

Loaded Stage-1: /Users/syedadnanahmad/Downloads/AI_TUMOUR_detection/models/stage1_balanced_best.pth
Loaded Stage-2: /Users/syedadnanahmad/Downloads/AI_TUMOUR_detection/models/oscc_topkK20_best.pth


In [5]:
np.random.seed(42); random.seed(42)
results = {}

# 1) Stage-2 multiclass evaluation (on OSCC test set) if model present
if model_stage2 is not None:
    print("\nEvaluating Stage-2 (oscc subtyping) on test split...")
    test_rows = df_index[(df_index["split"]=="test") & (df_index["class"].isin(OSCC_CLASSES))].reset_index(drop=True)
    rows_out = []
    y_true = []; y_pred = []; y_scores = []
    for _, r in test_rows.iterrows():
        slide = r["slide_id"]; cls = r["class"]
        emb_path = EMB_ROOT / "test" / cls / f"{slide}.npy"
        emb = load_npy(emb_path)
        emb_s = sample_K_from_emb(emb, K)
        H = torch.from_numpy(emb_s.astype(np.float32)).to(device)
        with torch.no_grad():
            logits, attn, idx = model_stage2(H)
            probs = F.softmax(logits, dim=1).cpu().numpy()[0]
        gt = OSCC_CLASSES.index(cls)
        pred = int(np.argmax(probs))
        rows_out.append({
            "slide_id": slide, "true_label": cls, "true_idx": gt,
            "pred_idx": pred, "pred_label": OSCC_CLASSES[pred],
            "probs": probs.tolist()
        })
        y_true.append(gt); y_pred.append(pred); y_scores.append(probs)
    df_out = pd.DataFrame(rows_out)
    df_out.to_csv(RESULTS_DIR / f"oscc_test_predictions_K{K}.csv", index=False)
    # metrics
    print(classification_report(y_true, y_pred, target_names=OSCC_CLASSES, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(OSCC_CLASSES))))
    plot_and_save_cm(cm, OSCC_CLASSES, RESULTS_DIR / f"oscc_cm_K{K}.png")
    # ROC AUC (one-vs-rest)
    y_scores = np.vstack(y_scores)
    aucs = safe_roc_auc(y_true, y_scores, n_classes=3, labels=[0,1,2])
    print("AUCs (oscc):", aucs)
    try:
        plot_roc_curves(y_true, y_scores, ["wdoscc","mdoscc","pdoscc"], RESULTS_DIR / f"oscc_roc_K{K}.png")
    except Exception as e:
        print("ROC plot error:", e)
    results["stage2"] = {"pred_csv": str(RESULTS_DIR / f"oscc_test_predictions_K{K}.csv"), "cm": str(RESULTS_DIR / f"oscc_cm_K{K}.png"), "aucs": aucs}



Evaluating Stage-2 (oscc subtyping) on test split...
              precision    recall  f1-score   support

      wdoscc       0.66      0.75      0.70        44
      mdoscc       0.60      0.43      0.50        42
      pdoscc       0.56      0.74      0.64        19

    accuracy                           0.62       105
   macro avg       0.61      0.64      0.61       105
weighted avg       0.62      0.62      0.61       105

AUCs (oscc): {0: 0.7704918032786885, 1: 0.7543461829176116, 2: 0.8451652386780907}


/var/folders/sf/m51mp7kn7773_zbvjf21qbph0000gn/T/ipykernel_10020/2941516752.py:53: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(loc="lower right")


In [6]:
# 2) Stage-1 binary evaluation 
if model_stage1 is not None:
    print("\nEvaluating Stage-1 (binary) on full test set...")
    test_rows = df_index[(df_index["split"]=="test") & (df_index["class"].isin(OSCC_CLASSES + ["normal"]))].reset_index(drop=True)
    rows_out = []
    y_true = []; y_pred = []; y_scores = []
    for _, r in test_rows.iterrows():
        slide = r["slide_id"]; cls = r["class"]
        emb_path = EMB_ROOT / "test" / cls / f"{slide}.npy"
        emb = load_npy(emb_path)
        emb_s = sample_K_from_emb(emb, K)
        H = torch.from_numpy(emb_s.astype(np.float32)).to(device)
        with torch.no_grad():
            logits, attn, idx = model_stage1(H)
            probs = F.softmax(logits, dim=1).cpu().numpy()[0]
        gt = 1 if cls in OSCC_CLASSES else 0
        pred = 1 if probs[1] > 0.65 else 0   # threshold used in training
        rows_out.append({
            "slide_id": slide, "true_label": ("oscc" if gt==1 else "normal"), "true_idx": gt,
            "pred_idx": pred, "pred_label": ("oscc" if pred==1 else "normal"),
            "probs": probs.tolist()
        })
        y_true.append(gt); y_pred.append(pred); y_scores.append(probs)
    df_out = pd.DataFrame(rows_out)
    df_out.to_csv(RESULTS_DIR / f"stage1_test_predictions_K{K}.csv", index=False)
    print(classification_report(y_true, y_pred, target_names=["normal","oscc"], zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    plot_and_save_cm(cm, ["normal","oscc"], RESULTS_DIR / f"stage1_cm_K{K}.png")
    y_scores = np.vstack(y_scores)
    aucs = safe_roc_auc(y_true, y_scores, n_classes=2, labels=[0,1])
    print("Stage1 AUCs:", aucs)
    try:
        plot_roc_curves(y_true, y_scores, ["normal","oscc"], RESULTS_DIR / f"stage1_roc_K{K}.png")
    except Exception as e:
        print("Stage1 ROC plot error:", e)
    results["stage1"] = {"pred_csv": str(RESULTS_DIR / f"stage1_test_predictions_K{K}.csv"), "cm": str(RESULTS_DIR / f"stage1_cm_K{K}.png"), "aucs": aucs}



Evaluating Stage-1 (binary) on full test set...
              precision    recall  f1-score   support

      normal       1.00      0.71      0.83        14
        oscc       0.96      1.00      0.98       105

    accuracy                           0.97       119
   macro avg       0.98      0.86      0.91       119
weighted avg       0.97      0.97      0.96       119

Stage1 AUCs: {0: 0.9829931972789115, 1: 0.9829931972789115}


/var/folders/sf/m51mp7kn7773_zbvjf21qbph0000gn/T/ipykernel_10020/2941516752.py:53: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(loc="lower right")


In [7]:
# 3) Combined pipeline evaluation (stage1->stage2) 
if model_stage1 is not None and model_stage2 is not None:
    print("\nEvaluating combined pipeline (stage1 -> stage2) on test set...")
    test_rows = df_index[(df_index["split"]=="test")].reset_index(drop=True)
    rows_out = []
    y_true = []; y_pred = []; y_scores = []
    for _, r in test_rows.iterrows():
        slide = r["slide_id"]; cls = r["class"]
        emb_path = EMB_ROOT / "test" / cls / f"{slide}.npy"
        emb = load_npy(emb_path)
        emb_s = sample_K_from_emb(emb, K)
        H = torch.from_numpy(emb_s.astype(np.float32)).to(device)
        with torch.no_grad():
            logits1, a1, i1 = model_stage1(H)
            p1 = F.softmax(logits1, dim=1).cpu().numpy()[0]
            pred1 = 1 if p1[1] > 0.65 else 0
            if pred1 == 1:
                logits2, a2, i2 = model_stage2(H)
                p2 = F.softmax(logits2, dim=1).cpu().numpy()[0]
                final_pred = ["wdoscc","mdoscc","pdoscc"][int(np.argmax(p2))]
                prob_vec = p2.tolist()
            else:
                final_pred = "normal"
                prob_vec = [None]
        rows_out.append({"slide_id": slide, "true_label": cls, "final_pred": final_pred, "prob_vec": prob_vec})
        y_true.append(cls); y_pred.append(final_pred); y_scores.append(prob_vec)
    pd.DataFrame(rows_out).to_csv(RESULTS_DIR / "combined_pipeline_predictions.csv", index=False)
    print("Combined pipeline predictions saved.")
    results["combined"] = {"pred_csv": str(RESULTS_DIR / "combined_pipeline_predictions.csv")}

# Save a summary JSON
with open(RESULTS_DIR / "evaluation_summary.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nDone. Results saved to:", RESULTS_DIR)


Evaluating combined pipeline (stage1 -> stage2) on test set...
Combined pipeline predictions saved.

Done. Results saved to: /Users/syedadnanahmad/Downloads/AI_TUMOUR_detection/results
